# Protein Secondary Structure Prediction: One‑Hot vs. ESM‑2 Embeddings

## Goal
This notebook trains a **simple MLP** to predict protein secondary structure (3‑state: helix `H`, sheet `E`, coil `C`) using two different sequence encoding methods:

1. **One‑hot encoding** – classical residue‑level representation.
2. **ESM‑2 embeddings** – deep contextualised features from a pretrained protein language model.

The performance of both encodings is compared on the same train/validation/test split.

## Dataset
**Source:** [Protein Secondary Structure (CASP12, CB513, TS115)](https://www.kaggle.com/datasets/tamzidhasan/protein-secondary-structure-casp12-cb513-ts115) by tamzidhasan.

The dataset contains three columns:
- `seq` – primary amino acid sequence (string).
- `sst3` – secondary structure in **3 states**:
  - `H` = helix (includes G, H, I)
  - `E` = sheet (includes B, E)
  - `C` = coil (includes T, S, C)
- `sst8` – secondary structure in **8 states** (not used here).

We use only `sst3` for 3‑class prediction. The data are already split into `train_df`, `valid_df`, `test_df` (e.g., from the original CASP12 / CB513 / TS115 partitions).
## Data Splits
- **Training** : proteins from CB513 + TS115 (or as provided)
- **Validation** : a subset of training (or separate)
- **Test** : CASP12 proteins (held‑out)
We load three pre‑split CSV files.

## Approach
1. **Window‑based feature extraction** – For each residue, we take a window of neighbouring residues (size = 15).  
   - One‑hot: each residue is encoded as a 20‑dim binary vector → window size × 20 features.  
   - ESM‑2: each residue is represented by a 1280‑dim embedding from the ESM‑2 (12‑layer, 35M parameter) model → window size × 1280 features.
2. **MLP architecture** – A simple fully‑connected network with two hidden layers (256 → 128) and dropout (0.3).
3. **Training** – Cross‑entropy loss, Adam optimiser, batch size 64, up to 20 epochs.
4. **Evaluation** – Accuracy on the test set.

## Why compare embeddings?
- **One‑hot** is fast, interpretable, but ignores evolutionary and structural context.
- **ESM‑2** captures long‑range dependencies and evolutionary information via pretraining on millions of protein sequences. It often gives higher accuracy, at the cost of computation.

The notebook provides a baseline (one‑hot) and a more advanced approach (ESM‑2) side‑by‑side, illustrating the benefit of deep learning based embeddings for protein secondary structure prediction.


In [9]:
# import libraries
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import os

# Import the ESM library
import esm

In [10]:
# define data root
data_root = '../../data/raw'

In [11]:
train_df = pd.read_csv(os.path.join(data_root, 'training_secondary_structure_train.csv'))
valid_df = pd.read_csv(os.path.join(data_root,'validation_secondary_structure_valid.csv'))
test_df = pd.read_csv(os.path.join(data_root,'test_secondary_structure_casp12.csv'))

ss3_map = {'H':0, 'E':1, 'C':2}
window_size = 15

print(f"Training proteins: {len(train_df)}")
print(f"Validation proteins: {len(valid_df)}")
print(f"Test (CASP12) proteins: {len(test_df)}")
print("Columns:", train_df.columns.tolist())

Training proteins: 8678
Validation proteins: 2170
Test (CASP12) proteins: 21
Columns: ['seq', 'sst3', 'sst8']


In [12]:
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path.cwd().parent.parent   # goes up two levels from models/protein_predict_mlp/
sys.path.append(str(project_root))

from embeddings import prepare_data_onehot, prepare_data_esm_chunked

## Training Function
- Optimizer: Adam
- Loss: Cross‑entropy
- Batch size: 64
- Epochs: 20 (early stopping not implemented, but we track best val acc)
- Evaluation: accuracy on test set after training
- 
## ESM‑2 Embeddings
We load `esm2_t12_35M_UR50D`, a 12‑layer, 35 million parameter ESM‑2 model.
This is the smallest variant, suitable for quick experiments. This model produces 1280‑dim per residue.
For a window of 15 residues, the input vector has 19200 dimensions → the MLP has many more parameters.
To avoid GPU memory overflow, we process proteins in chunks of 50 ( `chunk_size=50` ).

In [13]:
# ------------------------------------------------------------
# 3. MLP Model
# ------------------------------------------------------------
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, output_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, output_dim)
        )
    def forward(self, x):
        return self.net(x)


def get_default_device():
    """Return 'cuda' only if a compatible GPU is available."""
    if torch.cuda.is_available():
        # Check compute capability
        cap = torch.cuda.get_device_capability()
        if cap[0] >= 7:   # Requires at least sm_70
            return torch.device('cuda')
        else:
            print(f"GPU compute capability {cap[0]}.{cap[1]} is < 7.0. Falling back to CPU.")
    return torch.device('cpu')

# In train_and_evaluate, replace device detection with:
device = get_default_device()

# ------------------------------------------------------------
# 4. Training function with GPU support
# ------------------------------------------------------------
def train_and_evaluate(X_train, y_train, X_valid, y_valid, X_test, y_test,
                       input_dim, epochs=20, batch_size=64, lr=0.001, device=None):
    if device is None:
        device = get_default_device()
    print(f"Using device: {device}")
    
    model = SimpleMLP(input_dim).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    # Convert to tensors and move to device
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
    X_valid_t = torch.tensor(X_valid, dtype=torch.float32).to(device)
    y_valid_t = torch.tensor(y_valid, dtype=torch.long).to(device)
    
    train_dataset = TensorDataset(X_train_t, y_train_t)
    valid_dataset = TensorDataset(X_valid_t, y_valid_t)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)
    
    best_val_acc = 0
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for batch_x, batch_y in valid_loader:
                outputs = model(batch_x)
                _, preds = torch.max(outputs, 1)
                correct += (preds == batch_y).sum().item()
                total += batch_y.size(0)
        val_acc = correct / total
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1:2d} | Loss: {total_loss/len(train_loader):.4f} | Val Acc: {val_acc:.4f}")
    
    # Test evaluation
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    y_test_t = torch.tensor(y_test, dtype=torch.long).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_t)
        _, preds = torch.max(outputs, 1)
        test_acc = (preds == y_test_t).float().mean().item()
    print(f"Test Accuracy: {test_acc:.4f}")
    return test_acc, best_val_acc

# ------------------------------------------------------------
# 5. Run experiments (GPU automatically used if available)
# ------------------------------------------------------------
train_seqs = train_df['seq'].tolist()
train_ss3 = train_df['sst3'].tolist()
valid_seqs = valid_df['seq'].tolist()
valid_ss3 = valid_df['sst3'].tolist()
test_seqs = test_df['seq'].tolist()
test_ss3 = test_df['sst3'].tolist()

# ---- One-hot ----
print("=== One-hot encoding ===")
X_train_oh, y_train_oh = prepare_data_onehot(train_seqs, train_ss3, window_size)
X_valid_oh, y_valid_oh = prepare_data_onehot(valid_seqs, valid_ss3, window_size)
X_test_oh, y_test_oh   = prepare_data_onehot(test_seqs, test_ss3, window_size)
input_dim_oh = X_train_oh.shape[1]
acc_oh, val_oh = train_and_evaluate(X_train_oh, y_train_oh, X_valid_oh, y_valid_oh,
                                    X_test_oh, y_test_oh, input_dim_oh, epochs=15)

# ---- ESM ----
print("\n=== ESM embeddings ===")
subset_size = 200   # Using a subset for faster demonstration. Set to None to use all proteins.
if subset_size:
    train_seqs = train_df['seq'].tolist()[:subset_size]
    train_ss3 = train_df['sst3'].tolist()[:subset_size]
    valid_seqs = valid_df['seq'].tolist()[:subset_size]
    valid_ss3 = valid_df['sst3'].tolist()[:subset_size]
    test_seqs = test_df['seq'].tolist()[:subset_size]
    test_ss3 = test_df['sst3'].tolist()[:subset_size]
else:
    train_seqs = train_df['seq'].tolist()
    train_ss3 = train_df['sst3'].tolist()
    valid_seqs = valid_df['seq'].tolist()
    valid_ss3 = valid_df['sst3'].tolist()
    test_seqs = test_df['seq'].tolist()
    test_ss3 = test_df['sst3'].tolist()
    
device = get_default_device()          # respects capability
print(f"Using device for ESM: {device}")

esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()   #esm2_t12_35M_UR50D
esm_model = esm_model.to(device)
batch_converter = alphabet.get_batch_converter()


# Extract embeddings (this may take a few minutes)
# Prepare datasets
X_train_esm, y_train_esm = prepare_data_esm_chunked(
    train_seqs, train_ss3, esm_model, batch_converter, device, window_size, chunk_size=50)
X_valid_esm, y_valid_esm = prepare_data_esm_chunked(
    valid_seqs, valid_ss3, esm_model, batch_converter, device, window_size, chunk_size=50)
X_test_esm, y_test_esm = prepare_data_esm_chunked(
    test_seqs, test_ss3, esm_model, batch_converter, device, window_size, chunk_size=50)

# X_train_esm, y_train_esm = prepare_data_esm(train_seqs, train_ss3, esm_model, batch_converter, device, window_size)
# X_valid_esm, y_valid_esm = prepare_data_esm(valid_seqs, valid_ss3, esm_model, batch_converter, device, window_size)
# X_test_esm, y_test_esm   = prepare_data_esm(test_seqs, test_ss3, esm_model, batch_converter, device, window_size)
input_dim_esm = X_train_esm.shape[1]
acc_esm, val_esm = train_and_evaluate(X_train_esm, y_train_esm, X_valid_esm, y_valid_esm,
                                      X_test_esm, y_test_esm, input_dim_esm, epochs=15)

# ---- Comparison ----
print("\n=== Final Comparison ===")
print(f"One-hot encoding: Best Val = {val_oh:.4f}, Test = {acc_oh:.4f}")
print(f"ESM embeddings:  Best Val = {val_esm:.4f}, Test = {acc_esm:.4f}")

=== One-hot encoding ===
Using device: cpu
Epoch  5 | Loss: 0.7263 | Val Acc: 0.6915
Epoch 10 | Loss: 0.7187 | Val Acc: 0.6921
Epoch 15 | Loss: 0.7155 | Val Acc: 0.6929
Test Accuracy: 0.6719

=== ESM embeddings ===
Using device for ESM: cpu
Processing proteins 0 to 49...


ValueError: max() arg is an empty sequence

In [ ]:
# ------------------------------------------------------------
# 6. Detailed metrics: per‑class accuracy, confusion matrix, bar plot
# ------------------------------------------------------------
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def compute_metrics(model, X_test, y_test, device, embedding_name):
    """Compute predictions and return classification report & confusion matrix."""
    X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(X_test_t)
        _, preds = torch.max(outputs, 1)
    preds = preds.cpu().numpy()
    print(f"\n--- {embedding_name} ---")
    print(classification_report(y_test, preds, target_names=['H', 'E', 'C'], digits=4))
    cm = confusion_matrix(y_test, preds)
    return cm, preds

# Compute for one-hot
cm_oh, preds_oh = compute_metrics(model_oh, X_test_oh, y_test_oh, device, "One-hot")

# Compute for ESM
cm_esm, preds_esm = compute_metrics(model_esm, X_test_esm, y_test_esm, device, "ESM-2")

# Per‑class accuracy from classification report (or compute manually)
from sklearn.metrics import accuracy_score
classes = ['H', 'E', 'C']
per_class_acc_oh = []
per_class_acc_esm = []
for i, c in enumerate(classes):
    mask = (y_test_oh == i)
    per_class_acc_oh.append(accuracy_score(y_test_oh[mask], preds_oh[mask]))
    per_class_acc_esm.append(accuracy_score(y_test_esm[mask], preds_esm[mask]))

# Bar plot
x = np.arange(len(classes))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, per_class_acc_oh, width, label='One-hot', color='skyblue')
bars2 = ax.bar(x + width/2, per_class_acc_esm, width, label='ESM-2', color='salmon')

ax.set_ylabel('Accuracy')
ax.set_xlabel('Secondary Structure Class')
ax.set_title('Per‑class Accuracy Comparison')
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.legend()
ax.set_ylim(0, 1)

# Add value labels on bars
for bar in bars1 + bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

# Optional: plot confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm_oh, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title('Confusion Matrix - One-hot')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
sns.heatmap(cm_esm, annot=True, fmt='d', cmap='Oranges', xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title('Confusion Matrix - ESM-2')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
plt.tight_layout()
plt.show()